# Setup

In [ ]:
# set up snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE DATABASE LLM;
CREATE OR REPLACE SCHEMA LLM.TEST;
USE DATABASE LLM;
USE SCHEMA LLM.TEST;

# Introduction to Complete

In [ ]:
from snowflake.cortex import complete
complete("llama3.1-405b", "how do snowflakes get their unique patterns?")

# Introduction to Task-specific Functions

# Translate

In [ ]:
from snowflake.cortex import translate
translate("Snowflakes get their unique patterns through a combination of temperature, humidity, and air currents in the atmosphere.", "en", "fr")

In [ ]:
from snowflake.cortex import translate
translate("Les flocons de neige obtiennent leurs modèles uniques grâce à une combinaison de température, d’humidité et de courants atmosphériques.", "", "en")

In [ ]:
%%sql -r dataframe_9
CREATE or REPLACE file format csvformat
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  type = 'CSV';

CREATE or REPLACE stage call_transcripts_data_stage
  file_format = csvformat
  url = 's3://sfquickstarts/misc/call_transcripts/';

CREATE or REPLACE table CALL_TRANSCRIPTS ( 
  date_created date,
  language varchar(60),
  country varchar(60),
  product varchar(60),
  category varchar(60),
  damage_type varchar(90),
  transcript varchar
);

COPY into CALL_TRANSCRIPTS
  from @call_transcripts_data_stage;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE call_transcripts AS
SELECT *, SNOWFLAKE.CORTEX.TRANSLATE(transcript, '', 'en') AS EN_TRANSCRIPT
FROM call_transcripts;

SELECT * FROM call_transcripts LIMIT 10;

# Sentiment

In [ ]:
%%sql -r dataframe_3
SELECT transcript, 
    SNOWFLAKE.CORTEX.SENTIMENT(en_transcript) AS SENTIMENT
FROM call_transcripts
LIMIT 10;

# Summarize

In [ ]:
from snowflake.cortex import summarize

text = """
A snowflake is a single ice crystal that has achieved a sufficient size, and may have amalgamated with others, which falls through the Earth's atmosphere as snow.
Each flake nucleates around a tiny particle in supersaturated air masses by attracting supercooled cloud water droplets, which freeze and accrete in crystal form. 
Snow appears white in color despite being made of clear ice. 
This is due to diffuse reflection of the whole spectrum of light by the small crystal facets of the snowflakes.
"""

summarize(text)

# Text Classification

In [ ]:
%%sql -r dataframe_4
SELECT *,
SNOWFLAKE.CORTEX.CLASSIFY_TEXT(en_transcript, ['refund', 'replacement']) AS text_classify_output
FROM call_transcripts
LIMIT 50;

In [ ]:
%%sql -r dataframe_5
SELECT *,
SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
  en_transcript,
  [
    {
      'label': 'Refund',
      'description': 'questions or issues related to refunding the order amount',
      'examples': [
        'Hi, I noticed a defective product was delivered to me. I need a refund',
        'I received the wrong size of ski gear in my recent order. I would like to request a refund for this item.',
        'The ski gear delivered does not match the product description on your website. I need a refund.'
      ]
    },
    {
      'label': 'Replacement',
      'description': 'questions or issues related to replacing a product order',
      'examples': [
        'The snowboard I received was damaged during shipping. It has visible dents and scratches that affect its performance. I would appreciate it if you could send a replacement',
        'The ski goggles I received do not match the product description on your website. They lack the anti-fog coating that was advertised. Could you please provide a replacement that meets the described specifications?'
      ]
    }
  ],
  {'task_description': 'Return a classification of the type of issue described in the request'}
) AS text_classify_output
FROM call_transcripts;


In [ ]:
from snowflake.cortex import classify_text
classify_text("How can I buy a ticket for the Subway?", ["how to", "recommendations"])

In [ ]:
classify_text("What is the best broadway show to see right now?", ["how to", "recommendations"])

# Intro to Helper Functions

# TRY_COMPLETE()

In [ ]:
%%sql -r dataframe_6
SELECT SNOWFLAKE.CORTEX.TRY_COMPLETE(
    'llama2-70b-chaty',
    [
        {
            'role': 'user',
            'content': 'how does a snowflake get its unique pattern?'
        }
    ],
    {
        'temperature': 0.7,
        'max_tokens': 10
    }
);

# COUNT_TOKEN()

In [ ]:
%%sql -r dataframe_7
SELECT SNOWFLAKE.CORTEX.COUNT_TOKENS( 'llama3.1-70b', 
    'To be Jedi is to face the truth, and choose. Give off light, or darkness, Padawan. Be a candle, or the night.') 
    AS number_of_tokens;
